# Lección 4: Transformadores

Práctica de sesión 4. Atención desde cero sobre el párrafo de la sesión 3 y tokens de un modelo preentrenado.

---
<font size="3">
Christian Cabrera Jojoa<br>
Assistant Research Professor<br>
Department of Computer Science and Technology<br>
University of Cambridge<br>
chc79@cam.ac.uk
</font>

---
**Curso:** Machine Learning Tutorials<br>
**Departamento del curso:** Continuing-education workshops for civil engineering postgraduate students and teaching staff<br>
**Institución del curso:** Pontificia Universidad Católica del Perú (PUCP)


Este cuaderno construye la atención de la sesión desde cero y luego mira cómo un modelo real corta el texto en tokens. Cada bloque describe los datos utilizados, la tarea que se quiere solucionar y por qué el método encaja.

Ejecute las celdas de arriba hacia abajo. En cada **Comprobar**, confirme el mensaje `OK` antes de seguir. Las líneas marcadas con `# TODO` son las que usted escribe. Ya traen una solución para que el cuaderno corra. Bórrela, escriba la suya y vuelva a ejecutar el **Comprobar**.

La parte A corre bien en CPU. Para la parte opcional conviene una GPU: en Colab, Entorno de ejecución, Cambiar tipo de entorno, T4 GPU.

---

## Instalación

Colab ya trae casi todo. Esta celda solo asegura una versión de `transformers` que conoce el modelo de la parte B.

In [ ]:
%pip install -q "transformers>=4.56" requests pandas matplotlib

## Importaciones

En el siguiente bloque importamos las librerías que usaremos a lo largo del cuaderno.

In [ ]:
import math
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import torch
import torch.nn.functional as F
from torch import nn
from transformers import AutoModel, AutoTokenizer

---

## Parte A: atención desde cero

Usamos el párrafo de la sesión 3 y la misma tarea: leer 11 caracteres y predecir el que sigue. La red recurrente leía la ventana de uno en uno. Una cabeza de atención mira los 11 caracteres a la vez y decide cuánto pesa cada uno.

Construimos la atención en tres pasos: puntajes escalados, máscara causal y softmax. Luego entrenamos una cabeza y la comparamos con la red recurrente.

### El párrafo

El texto cabe en el cuaderno. Es corto a propósito, para que una red pequeña pueda aprenderlo en la sesión.

In [ ]:
texto = (
    "el temblor movió la estación cercana. "
    "la estación registró el movimiento del suelo. "
    "el operador revisó la señal y avisó al centro. "
    "el centro comparó la señal con las estaciones vecinas."
)
print(texto)
print("largo:", len(texto))

Cada fila es una ventana de 11 caracteres, y el objetivo es el carácter que sigue. La primera ventana es `el temblor` más el espacio que la separa de `movió`. El objetivo de esa fila es la `m`. La segunda se corre un carácter y el objetivo es la `o`. La tercera empieza con un espacio y el objetivo es la `v`. La última es `nes vecinas` y el objetivo es el punto final.

En la función, `texto[i - 11:i]` copia esos 11 caracteres. Cada carácter se vuelve un vector de ceros con un 1 en su columna del alfabeto. `np.stack` apila esas ventanas. Esta transformación permite comparar indices (i.e., posiciones en el alfabeto) en lugar del carácter escrito.

In [ ]:
def preparar_texto(texto, ventana=11):
    """Arma ventanas de caracteres y el índice del carácter que sigue.

    Args:
        texto: Párrafo del que salen las ventanas.
        ventana: Cuántos caracteres mira el modelo antes de predecir el siguiente.

    Returns:
        Las ventanas en one-hot, el índice de cada objetivo y el alfabeto.
    """
    # Ordena los caracteres distintos del párrafo
    alfabeto = sorted(set(texto))
    # Crea un diccionario para convertir un caracter en su indice
    indice = {caracter: i for i, caracter in enumerate(alfabeto)}
    filas = []
    objetivos = []
    for i in range(ventana, len(texto)):
        # 11 filas (i.e., caracteres) y una columna por letra del alfabeto
        fila = np.zeros((ventana, len(alfabeto)))
        for t, caracter in enumerate(texto[i - ventana:i]):
            # Pone el 1 en la columna de ese carácter
            fila[t, indice[caracter]] = 1.0
        filas.append(fila)
        # Guarda el índice del carácter que sigue a la ventana
        objetivos.append(indice[texto[i]])
    # Apila las ventanas en un solo bloque
    return np.stack(filas), np.array(objetivos), alfabeto

Cada ventana es una matriz de 11 filas y 22 columnas. Cada fila es un carácter. Cada columna es una letra del alfabeto. Hay un 1 en la columna de ese carácter y 0 en las demás. La `e` de la primera ventana tiene el 1 en la columna 6. La red no lee las letras. Lee estas matrices, así que esa es la entrada. `X_texto` apila las 174 matrices, una por ventana. En PyTorch ese bloque se llama tensor y su forma se escribe `(174, 11, 22)`. `y_texto` es el índice del carácter objetivo, uno por ventana, y no es parte de esas matrices.

In [ ]:
X_texto, y_texto, alfabeto = preparar_texto(texto)
print("ventanas", X_texto.shape, "| alfabeto", len(alfabeto))
for nombre, k in (("primera", 0), ("segunda", 1), ("tercera", 2), ("última", -1)):
    # Lee la columna del 1 en cada fila y la vuelve a convertir en letra
    ventana = "".join(alfabeto[i] for i in X_texto[k].argmax(axis=1))
    print(nombre, repr(ventana), "| objetivo:", repr(alfabeto[y_texto[k]]))

### Puntajes escalados

Cada carácter produce tres vectores. La consulta $\mathbf{q}$ dice qué busca, la clave $\mathbf{k}$ dice qué ofrece y el valor $\mathbf{v}$ es lo que entrega. El puntaje entre las posiciones $i$ y $j$ es $\mathbf{q}_i \cdot \mathbf{k}_j / \sqrt{d}$. Dividir por $\sqrt{d}$ evita que los puntajes crezcan con el tamaño $d$ del vector.

`K.transpose(-2, -1)` gira las dos últimas dimensiones de `K`. Así `Q @ K.transpose(-2, -1)` calcula todos los productos de una vez, para una ventana o para un lote de ventanas.

In [ ]:
def puntajes_escalados(Q, K):
    """Producto de cada consulta con cada clave, dividido por la raíz de d.

    Args:
        Q: Consultas, forma (..., T, d).
        K: Claves, forma (..., T, d).

    Returns:
        Puntajes de forma (..., T, T).
    """
    d = K.shape[-1]
    # TODO: multiplique Q por K transpuesta y divida por la raíz de d
    return Q @ K.transpose(-2, -1) / math.sqrt(d)

### Máscara causal

Para predecir lo que sigue, la posición $i$ solo puede mirar las posiciones $0$ a $i$. La máscara marca con `True` todo lo que queda a la derecha de la diagonal. Esos puntajes pasan a $-\infty$ y, después de la softmax, pesan cero.

In [ ]:
def mascara_causal(T):
    """Marca las posiciones futuras de una ventana de largo T.

    Args:
        T: Largo de la ventana.

    Returns:
        Tensor booleano (T, T), True arriba de la diagonal.
    """
    # TODO: triangular superior de unos, sin la diagonal
    return torch.triu(torch.ones(T, T, dtype=torch.bool), diagonal=1)

### Softmax

La softmax convierte cada fila de puntajes en pesos positivos que suman 1. Restamos el máximo de la fila antes de `exp`. El resultado no cambia y se evitan números enormes. Un $-\infty$ se vuelve $e^{-\infty} = 0$.

In [ ]:
def softmax_filas(puntajes):
    """Softmax sobre la última dimensión.

    Args:
        puntajes: Tensor de forma (..., T, T).

    Returns:
        Pesos de la misma forma. Cada fila suma 1.
    """
    # TODO: reste el máximo de cada fila, aplique exp y divida por la suma de la fila
    exponencial = torch.exp(puntajes - puntajes.max(dim=-1, keepdim=True).values)
    return exponencial / exponencial.sum(dim=-1, keepdim=True)

### La función completa

`toy_attention` junta los tres pasos. Devuelve la mezcla de valores y los pesos, para poder dibujarlos después.

In [ ]:
def toy_attention(Q, K, V, causal=True):
    """Atención de una cabeza: puntajes escalados, máscara causal y softmax.

    Args:
        Q: Consultas, forma (..., T, d).
        K: Claves, forma (..., T, d).
        V: Valores, forma (..., T, d).
        causal: Si es True, cada posición solo mira hacia atrás.

    Returns:
        La mezcla de valores (..., T, d) y los pesos (..., T, T).
    """
    puntajes = puntajes_escalados(Q, K)
    if causal:
        # Las posiciones futuras quedan en menos infinito
        puntajes = puntajes.masked_fill(mascara_causal(Q.shape[-2]), float("-inf"))
    pesos = softmax_filas(puntajes)
    return pesos @ V, pesos

Probamos con cuatro posiciones al azar. PyTorch trae su propia atención, `scaled_dot_product_attention`. Si la nuestra está bien, las dos salidas coinciden.

In [ ]:
torch.manual_seed(0)
Q, K, V = torch.randn(4, 8), torch.randn(4, 8), torch.randn(4, 8)
salida, pesos = toy_attention(Q, K, V)
referencia = F.scaled_dot_product_attention(Q[None], K[None], V[None], is_causal=True)[0]
print(pesos.round(decimals=2))

**Comprobar:**

In [ ]:
assert torch.allclose(pesos.sum(dim=-1), torch.ones(4))
assert torch.all(pesos[mascara_causal(4)] == 0)
assert torch.allclose(salida, referencia, atol=1e-5)
print("La atención coincide con la de PyTorch. OK")

### Una cabeza de atención

La entrada es la misma matriz one-hot de 11 por 22. Una capa lineal convierte cada carácter en un vector de 32 números. Un vector aprendido por posición le suma dónde está en la ventana. Sin él, la atención no distingue el orden.

Tres capas lineales sacan $Q$, $K$ y $V$. La atención mezcla los valores, y la suma con la entrada es la conexión residual. Una red pequeña por posición (feed-forward) procesa esa mezcla. La cabeza lineal lee solo la última posición, la única que vio los 11 caracteres, y puntúa el carácter siguiente.

In [ ]:
class CabezaAtencion(nn.Module):
    """Una cabeza de atención causal que puntúa el carácter siguiente.

    Args:
        n_caracteres: Tamaño del alfabeto. Es también el tamaño de cada one-hot.
        d: Tamaño del vector de cada carácter.
        ventana: Largo de la ventana.
    """

    def __init__(self, n_caracteres, d=32, ventana=11):
        super().__init__()
        self.entrada = nn.Linear(n_caracteres, d)
        # Un vector aprendido por posición
        self.posicion = nn.Parameter(0.1 * torch.randn(ventana, d))
        self.q = nn.Linear(d, d, bias=False)
        self.k = nn.Linear(d, d, bias=False)
        self.v = nn.Linear(d, d, bias=False)
        self.feed_forward = nn.Sequential(nn.Linear(d, 4 * d), nn.ReLU(), nn.Linear(4 * d, d))
        self.cabeza = nn.Linear(d, n_caracteres)

    def forward(self, x):
        """Puntúa el carácter siguiente a cada ventana.

        Args:
            x: Tensor de forma (lote, caracteres, alfabeto).

        Returns:
            Puntajes (lote, alfabeto), sin softmax, y pesos (lote, caracteres, caracteres).
        """
        h = self.entrada(x) + self.posicion
        mezcla, pesos = toy_attention(self.q(h), self.k(h), self.v(h))
        # Conexión residual: la mezcla se suma a la entrada
        h = h + mezcla
        h = h + self.feed_forward(h)
        # La última posición ya miró toda la ventana
        return self.cabeza(h[:, -1]), pesos

El entrenamiento es el de la sesión 3: entropía cruzada, Adam y 400 épocas. La tasa de aprendizaje es más alta porque la red es más pequeña.

In [ ]:
def entrenar_atencion(X, y, d=32, epocas=400):
    """Ajusta la cabeza de atención para predecir el índice del carácter siguiente.

    Args:
        X: Ventanas one-hot, forma (filas, caracteres, alfabeto).
        y: Índice del carácter objetivo, uno por ventana.
        d: Tamaño del vector de cada carácter.
        epocas: Pasadas completas por el párrafo.

    Returns:
        El modelo entrenado y la pérdida de cada época.
    """
    torch.manual_seed(42)
    lote_x = torch.tensor(X, dtype=torch.float32)
    lote_y = torch.tensor(y, dtype=torch.long)
    modelo = CabezaAtencion(n_caracteres=X.shape[-1], d=d, ventana=X.shape[1])
    optimizador = torch.optim.Adam(modelo.parameters(), lr=1e-2)
    perdida_fn = nn.CrossEntropyLoss()
    historial = []
    for epoca in range(epocas):
        optimizador.zero_grad()
        puntajes, _ = modelo(lote_x)
        perdida = perdida_fn(puntajes, lote_y)
        perdida.backward()
        optimizador.step()
        historial.append(perdida.item())
    return modelo, historial

### La red recurrente

La red lee los 11 caracteres de uno en uno. `batch_first=True` dice que cada matriz de 11 por 22 entra como una ventana, con los 11 caracteres en orden. Es el orden en que `np.stack` apiló las filas. En cada carácter actualiza un estado oculto de 128 números, con los mismos pesos. `salida` guarda el estado de cada carácter. `h_T[-1]` es solo el estado del último carácter, porque hay una sola capa. La cabeza lineal convierte esos 128 números en un puntaje por carácter del alfabeto.

En el primer paso lee la `e` de `el temblor`. En el segundo lee la `l`, con los mismos pesos, y así hasta el espacio. El puntaje más alto después de ese espacio es la predicción de la `m`.

In [ ]:
class ModeloTexto(nn.Module):
    """Lee una ventana carácter por carácter y puntúa el carácter siguiente.

    El estado oculto se actualiza en cada paso con los mismos pesos.

    Args:
        n_caracteres: Tamaño del alfabeto. Es también el tamaño de cada one-hot.
        n_ocultas: Tamaño del estado oculto.
    """

    def __init__(self, n_caracteres, n_ocultas):
        super().__init__()
        self.rnn = nn.RNN(n_caracteres, n_ocultas, batch_first=True)
        self.cabeza = nn.Linear(n_ocultas, n_caracteres)

    def forward(self, x):
        """Puntúa el carácter siguiente a cada ventana.

        Args:
            x: Tensor de forma (lote, caracteres, alfabeto).

        Returns:
            Tensor de forma (lote, alfabeto), sin softmax.
        """
        # Guarda el estado de cada carácter y el estado final de la única capa
        salida, h_T = self.rnn(x)
        # Convierte el estado del último carácter en un puntaje por letra
        return self.cabeza(h_T[-1])

La pérdida es la entropía cruzada entre esos puntajes y el índice del carácter correcto. Adam ajusta el paso, y 400 épocas le dan tiempo a aprender un párrafo tan corto. El recorte del gradiente limita el producto de pendientes a lo largo de los 11 caracteres.

In [ ]:
def entrenar_texto(X, y, n_ocultas=128, epocas=400):
    """Ajusta la red para predecir el índice del carácter siguiente.

    Args:
        X: Ventanas one-hot, forma (filas, caracteres, alfabeto).
        y: Índice del carácter objetivo, uno por ventana.
        n_ocultas: Tamaño del estado oculto.
        epocas: Pasadas completas por el párrafo.

    Returns:
        El modelo entrenado y la pérdida de cada época.
    """
    torch.manual_seed(42)
    lote_x = torch.tensor(X, dtype=torch.float32)
    # El objetivo entra como índice entero
    lote_y = torch.tensor(y, dtype=torch.long)
    # La última dimensión de X es el tamaño del alfabeto
    modelo = ModeloTexto(n_caracteres=X.shape[-1], n_ocultas=n_ocultas)
    optimizador = torch.optim.Adam(modelo.parameters(), lr=1e-3)
    perdida_fn = nn.CrossEntropyLoss()
    historial = []
    for epoca in range(epocas):
        optimizador.zero_grad()
        puntajes = modelo(lote_x)
        # Compara los puntajes con el índice del carácter correcto
        perdida = perdida_fn(puntajes, lote_y)
        perdida.backward()
        # Después de backward cada peso tiene un gradiente.
        # Si el largo de ese vector pasa de 1, lo encoge hasta dejarlo en 1.
        # El producto de las pendientes en los 11 caracteres puede hacer enorme el paso.
        nn.utils.clip_grad_norm_(modelo.parameters(), max_norm=1.0)
        optimizador.step()
        historial.append(perdida.item())
    return modelo, historial

### Atención frente a la red recurrente

Entrenamos las dos redes con las mismas ventanas y medimos cuatro cosas: parámetros, pérdida final, exactitud sobre el párrafo y segundos de entrenamiento. La exactitud es sobre el mismo texto que vieron, así que mide cuánto lo memorizaron, no cuánto generalizan.

In [ ]:
def exactitud(modelo, X, y):
    """Fracción de ventanas donde el puntaje más alto es el carácter correcto.

    Args:
        modelo: Red ya entrenada.
        X: Ventanas one-hot.
        y: Índice del carácter objetivo.

    Returns:
        La exactitud entre 0 y 1.
    """
    modelo.eval()
    with torch.no_grad():
        salida = modelo(torch.tensor(X, dtype=torch.float32))
    # La cabeza de atención devuelve también los pesos
    if isinstance(salida, tuple):
        salida = salida[0]
    return float((salida.argmax(dim=1).numpy() == y).mean())


inicio = time.perf_counter()
atencion_texto, historial_atencion = entrenar_atencion(X_texto, y_texto)
segundos_atencion = time.perf_counter() - inicio
inicio = time.perf_counter()
rnn_texto, historial_texto = entrenar_texto(X_texto, y_texto)
segundos_rnn = time.perf_counter() - inicio

comparacion = pd.DataFrame(
    {
        "parámetros": [sum(p.numel() for p in m.parameters()) for m in (atencion_texto, rnn_texto)],
        "pérdida final": [historial_atencion[-1], historial_texto[-1]],
        "exactitud": [exactitud(atencion_texto, X_texto, y_texto), exactitud(rnn_texto, X_texto, y_texto)],
        "segundos": [segundos_atencion, segundos_rnn],
    },
    index=["atención", "recurrente"],
)
print(comparacion.round(3))

plt.figure(figsize=(5, 3.5))
plt.plot(historial_atencion, label="atención")
plt.plot(historial_texto, label="recurrente")
plt.xlabel("época")
plt.ylabel("pérdida")
plt.yscale("log")
plt.legend()
plt.show()

Las dos redes aprenden el párrafo. La diferencia está en cómo leen. La recurrente da 11 pasos en orden, y cada paso espera al anterior. La atención calcula las 11 posiciones con unas pocas multiplicaciones de matrices, que es justo lo que una GPU hace rápido. Con 174 ventanas casi no se nota, y en CPU la recurrente puede incluso terminar antes. Con millones de tokens es la razón por la que los transformadores reemplazaron a las redes recurrentes.

**Comprobar:**

In [ ]:
assert historial_atencion[-1] < historial_atencion[0]
assert comparacion.loc["atención", "exactitud"] > 0.9
assert comparacion.loc["atención", "parámetros"] < comparacion.loc["recurrente", "parámetros"]
print("La cabeza de atención aprende el párrafo. OK")

### El mapa de atención

Cada fila es una posición de la ventana, y cada columna es un carácter que esa posición puede mirar. Arriba de la diagonal todo es cero por la máscara. La última fila es la que usa la cabeza para predecir. Los espacios se dibujan como `␣`.

In [ ]:
def mapa_atencion(pesos, etiquetas, titulo):
    """Dibuja una matriz de pesos de atención.

    Args:
        pesos: Matriz (T, T). La fila i son los pesos de la posición i.
        etiquetas: Texto de cada posición.
        titulo: Título del gráfico.
    """
    fig, ax = plt.subplots(figsize=(5.5, 5))
    imagen = ax.imshow(pesos, cmap="viridis", vmin=0, vmax=1)
    ax.set_xticks(range(len(etiquetas)), etiquetas, rotation=90)
    ax.set_yticks(range(len(etiquetas)), etiquetas)
    ax.set_xlabel("posición mirada (clave)")
    ax.set_ylabel("posición que mira (consulta)")
    ax.set_title(titulo)
    fig.colorbar(imagen, ax=ax)
    plt.show()

`k = 0` es la primera ventana, `el temblor` y un espacio, con objetivo `m`. Cambie `k` para ver otras ventanas.

In [ ]:
k = 0
with torch.no_grad():
    puntajes_k, pesos_k = atencion_texto(torch.tensor(X_texto[k:k + 1], dtype=torch.float32))
caracteres_k = [alfabeto[i].replace(" ", "␣") for i in X_texto[k].argmax(axis=1)]
print("ventana:", repr("".join(alfabeto[i] for i in X_texto[k].argmax(axis=1))),
      "| predicción:", repr(alfabeto[int(puntajes_k.argmax())]),
      "| objetivo:", repr(alfabeto[y_texto[k]]))
mapa_atencion(pesos_k[0].numpy(), caracteres_k, f"Cabeza entrenada, ventana {k}")

**Comprobar:**

In [ ]:
assert pesos_k.shape == (1, 11, 11)
assert torch.all(pesos_k[0][mascara_causal(11)] == 0)
assert torch.allclose(pesos_k.sum(dim=-1), torch.ones(1, 11))
print("El mapa respeta la máscara causal. OK")

---

## Parte B: tokens

Un transformador real no lee caracteres. Lee tokens, piezas de palabra de un vocabulario fijo. Usamos el tokenizador de `Qwen/Qwen2.5-0.5B-Instruct`, un modelo abierto pequeño que maneja español. De este modelo solo usamos el tokenizador y, en la parte opcional, sus pesos de atención. No generamos texto.

La tarea es contar los tokens de un texto frente a sus caracteres y sus palabras, y ver dónde se cortan las palabras. Importa porque el largo máximo y el costo de un modelo se miden en tokens, no en palabras.

### El tokenizador

`AutoTokenizer` descarga el tokenizador desde [Hugging Face](https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct) la primera vez. Son unos pocos megabytes. Su vocabulario se armó con pares de bytes (BPE): las secuencias frecuentes quedan como una sola pieza y las raras se arman con piezas más cortas.

In [ ]:
MODELO_HF = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizador = AutoTokenizer.from_pretrained(MODELO_HF)
print("tamaño del vocabulario:", len(tokenizador))

`tokenizador(texto).input_ids` da el número de cada token. `piezas` decodifica cada número por separado para ver el texto de cada pieza. Un espacio inicial dentro de una pieza es parte del token.

In [ ]:
def piezas(texto):
    """Texto de cada token, en orden.

    Args:
        texto: Cadena a tokenizar.

    Returns:
        Lista con el texto de cada token.
    """
    return [tokenizador.decode([i]) for i in tokenizador(texto).input_ids]

### Contar tokens

Escriba `contar`. Las palabras salen de `texto.split()`, que corta en los espacios.

In [ ]:
def contar(texto):
    """Cuenta caracteres, palabras y tokens de un texto.

    Args:
        texto: Cadena a contar.

    Returns:
        Un diccionario con las tres cuentas.
    """
    # TODO: use len, texto.split() y el tokenizador
    return {
        "caracteres": len(texto),
        "palabras": len(texto.split()),
        "tokens": len(tokenizador(texto).input_ids),
    }

Primero el párrafo de la parte A. La red de la parte A veía caracteres. Qwen ve muchas menos piezas, porque muchas palabras comunes son un solo token.

In [ ]:
print(contar(texto))
print(piezas(texto.split(". ")[0] + "."))

### Los lugares del catálogo

La columna `place` del catálogo USGS describe dónde fue cada sismo, por ejemplo `45 km SW of Acarí, Peru`. Descargamos el recorte del Perú de la sesión 1, 2024 y magnitud 4.5 o más, y tokenizamos esa columna.

`build_usgs_url` y `download_csv` son las funciones de la sesión 1. La primera arma la URL del catálogo USGS a partir de un diccionario. La segunda descarga el CSV y falla temprano si la respuesta no es el catálogo.

In [ ]:
def build_usgs_url(parametros: dict) -> str:
    """Arma la URL del catálogo USGS a partir de un diccionario de parámetros."""
    base = "https://earthquake.usgs.gov/fdsnws/event/1/query"
    consulta = "&".join(f"{clave}={valor}" for clave, valor in parametros.items())
    return f"{base}?{consulta}"


def download_csv(url: str, dest: Path, timeout: int = 120) -> int:
    """Descarga url a dest y devuelve el número de bytes.

    Falla si la respuesta no es el CSV del catálogo.
    """
    respuesta = requests.get(url, timeout=timeout)
    respuesta.raise_for_status()  # falla si el servidor respondió 4xx o 5xx
    texto = respuesta.text
    if not texto.startswith("time,"):  # el CSV del catálogo siempre empieza así
        raise ValueError(
            "La respuesta no es el CSV del catálogo. "
            "Revise la URL antes de seguir."
        )
    dest.write_text(texto, encoding="utf-8")
    return dest.stat().st_size

`PARAMETROS_CURSO` es el mismo recorte de las sesiones anteriores. No lo modifique.

In [ ]:
PARAMETROS_CURSO = {
    "format": "csv",
    "starttime": "2024-01-01",
    "endtime": "2025-01-01",
    "minmagnitude": "4.5",
    "minlatitude": "-18.5",
    "maxlatitude": "0.2",
    "minlongitude": "-82",
    "maxlongitude": "-68",
}
CATALOGO_PERU = Path("usgs_peru_2024_m45.csv")

download_csv(build_usgs_url(PARAMETROS_CURSO), CATALOGO_PERU)
sismos_peru = pd.read_csv(CATALOGO_PERU)
lugares = sismos_peru["place"].dropna()
print("sismos:", len(sismos_peru), "| lugares:", len(lugares))
print(lugares.head().to_list())

Contamos cada lugar y calculamos tokens por palabra. Un valor cercano a 1 quiere decir que casi todas las palabras son un solo token.

In [ ]:
conteos = pd.DataFrame([contar(lugar) for lugar in lugares], index=lugares.index)
conteos.insert(0, "place", lugares)
conteos["tokens por palabra"] = conteos["tokens"] / conteos["palabras"]
print(conteos[["caracteres", "palabras", "tokens"]].sum())
print(conteos.sort_values("tokens por palabra", ascending=False).head().round(2).to_string())

### Dónde se cortan las palabras

Miramos las piezas de un lugar completo y de algunas palabras sueltas. Dentro de la frase, ` km`, ` of` y ` Peru` son un solo token cada uno, con su espacio delante. Son frecuentes en el texto con que se armó el vocabulario. Los nombres de lugares peruanos no.

In [ ]:
for palabra in ["45 km SW of Acarí, Peru", "Acarí", "Chinchaypujio", "45", "2024"]:
    print(f"{palabra!r:28} -> {piezas(palabra)}")

`Acarí` se parte en tres piezas, y la `í` queda sola. Qwen escribe cada dígito como un token aparte, así que el modelo no ve el número 45. Ve un 4 seguido de un 5. Pruebe con otros lugares de la tabla o con un texto suyo.

**Comprobar:**

In [ ]:
assert len(piezas("Acarí")) > 1
assert piezas("2024") == ["2", "0", "2", "4"]
assert (conteos["tokens"] >= conteos["palabras"]).all()
print("Los tokens parten las palabras raras y los números. OK")

---

## Parte opcional: la atención de un modelo preentrenado

Si le queda tiempo, mire los pesos de atención del mismo modelo sobre la primera frase del párrafo. Es la misma operación de la parte A, aprendida con mucho más texto.

`AutoModel` carga el transformador sin la capa que predice tokens, unos 1 GB de descarga. Con GPU usamos `float16`, que ocupa la mitad de memoria. Sin GPU el mismo código corre en CPU con `float32`, solo más lento. `attn_implementation="eager"` pide la atención escrita paso a paso, la que devuelve los pesos.

In [ ]:
dispositivo = "cuda" if torch.cuda.is_available() else "cpu"
tipo = torch.float16 if dispositivo == "cuda" else torch.float32
modelo_hf = AutoModel.from_pretrained(MODELO_HF, dtype=tipo, attn_implementation="eager").to(dispositivo)
modelo_hf.eval()
print("dispositivo:", dispositivo,
      "| capas:", modelo_hf.config.num_hidden_layers,
      "| cabezas por capa:", modelo_hf.config.num_attention_heads)

`torch.no_grad()` evita guardar gradientes, porque solo leemos el modelo. `output_attentions=True` devuelve una matriz de pesos por capa y por cabeza. `last_hidden_state` es el vector final de cada token.

In [ ]:
frase = texto.split(". ")[0] + "."
entrada = tokenizador(frase, return_tensors="pt").to(dispositivo)
with torch.no_grad():
    salida_hf = modelo_hf(**entrada, output_attentions=True)
print("tokens:", entrada["input_ids"].shape[1],
      "| vectores finales:", tuple(salida_hf.last_hidden_state.shape))

Elija una capa y una cabeza. La primera capa es 0. En la capa 9, cabeza 5, las piezas de `movió` miran a las de `temblor`. Pruebe luego la capa 6, cabeza 0, que pone casi todo el peso en el primer token. Muchas cabezas hacen eso.

In [ ]:
CAPA, CABEZA = 9, 5
pesos_hf = salida_hf.attentions[CAPA][0, CABEZA].float().cpu()
mapa_atencion(pesos_hf.numpy(), piezas(frase), f"Qwen2.5-0.5B, capa {CAPA}, cabeza {CABEZA}")

**Comprobar:**

In [ ]:
assert len(salida_hf.attentions) == modelo_hf.config.num_hidden_layers
assert torch.all(pesos_hf[mascara_causal(len(pesos_hf))] == 0)
assert torch.allclose(pesos_hf.sum(dim=-1), torch.ones(len(pesos_hf)), atol=1e-2)
print("El modelo usa la misma máscara causal. OK")